In [6]:
import pandas as pd

df_fatture = pd.read_csv("fatture_aperte.csv", parse_dates=["data_emissione", "data_scadenza"])

# Data di riferimento dell'analisi: "oggi" per l'azienda simulata
oggi = pd.Timestamp("2026-07-03")

print(f"Fatture totali: {len(df_fatture)}")
df_fatture.head()

Fatture totali: 13


,numero_fattura,cliente,importo,data_emissione,data_scadenza,numero
0,FT-2026-006,AgriToscana SOC COOP,1154.42,2026-01-07,2026-02-06,6
1,FT-2026-010,Meccanica Fiorentina SRL,946.30,2026-03-29,2026-04-28,10
2,FT-2026-012,Studio Neri & Associati,4267.97,2026-01-25,2026-02-24,12
3,FT-2026-015,Bianchi Alimentari SPA,2849.80,2026-04-07,2026-05-07,15
4,FT-2026-017,Rossi Costruzioni SRL,3374.06,2026-01-18,2026-02-17,17


In [7]:
# Giorni di ritardo rispetto alla scadenza (negativo = non ancora scaduta)
df_fatture["giorni_ritardo"] = (oggi - df_fatture["data_scadenza"]).dt.days

# Assegnazione della fascia di aging
def fascia_aging(giorni):
    if giorni <= 0:
        return "Non scaduta"
    elif giorni <= 30:
        return "0-30 giorni"
    elif giorni <= 60:
        return "31-60 giorni"
    elif giorni <= 90:
        return "61-90 giorni"
    else:
        return "Oltre 90 giorni"

df_fatture["fascia"] = df_fatture["giorni_ritardo"].apply(fascia_aging)

df_fatture[["numero_fattura", "cliente", "importo", "data_scadenza", "giorni_ritardo", "fascia"]].head(10)

,numero_fattura,cliente,importo,data_scadenza,giorni_ritardo,fascia
0,FT-2026-006,AgriToscana SOC COOP,1154.42,2026-02-06,147,Oltre 90 giorni
1,FT-2026-010,Meccanica Fiorentina SRL,946.30,2026-04-28,66,61-90 giorni
2,FT-2026-012,Studio Neri & Associati,4267.97,2026-02-24,129,Oltre 90 giorni
3,FT-2026-015,Bianchi Alimentari SPA,2849.80,2026-05-07,57,31-60 giorni
4,FT-2026-017,Rossi Costruzioni SRL,3374.06,2026-02-17,136,Oltre 90 giorni
5,FT-2026-019,Elettrodomestici Gallo SRL,1534.27,2026-02-25,128,Oltre 90 giorni
6,FT-2026-029,Toscana Impianti SNC,3346.11,2026-04-21,73,61-90 giorni
7,FT-2026-044,AgriToscana SOC COOP,3857.42,2026-04-08,86,61-90 giorni
8,FT-2026-047,Studio Neri & Associati,3860.05,2026-03-11,114,Oltre 90 giorni
9,FT-2026-048,AgriToscana SOC COOP,4608.77,2026-06-18,15,0-30 giorni


In [8]:
print(df_fatture["fascia"].value_counts())
print(f"\nRitardo minimo: {df_fatture['giorni_ritardo'].min()} giorni")
print(f"Ritardo massimo: {df_fatture['giorni_ritardo'].max()} giorni")
print(f"\nScadenza più vecchia: {df_fatture['data_scadenza'].min()}")
print(f"Scadenza più recente: {df_fatture['data_scadenza'].max()}")

fascia
Oltre 90 giorni    7
61-90 giorni       3
31-60 giorni       2
0-30 giorni        1
Name: count, dtype: int64

Ritardo minimo: 15 giorni
Ritardo massimo: 147 giorni

Scadenza più vecchia: 2026-02-06 00:00:00
Scadenza più recente: 2026-06-18 00:00:00


In [9]:
aging = df_fatture.groupby("fascia").agg(
    numero_fatture=("numero_fattura", "count"),
    importo_totale=("importo", "sum")
).reindex(["Non scaduta", "0-30 giorni", "31-60 giorni", "61-90 giorni", "Oltre 90 giorni"])

totale = aging["importo_totale"].sum()
aging["percentuale"] = (aging["importo_totale"] / totale * 100).round(1)

print(f"Crediti aperti totali: {totale:,.2f} €")
aging

Crediti aperti totali: 39,362.41 €


,numero_fatture,importo_totale,percentuale
fascia,,,
Non scaduta,NaN,NaN,NaN
0-30 giorni,1.0,4608.77,11.7
31-60 giorni,2.0,3842.41,9.8
61-90 giorni,3.0,8149.83,20.7
Oltre 90 giorni,7.0,22761.40,57.8
